# Active Ingredient Cleaning & Verification Pipeline

Two-stage pipeline: (1) clean and normalize ingredient names, (2) verify against reference data and LLM.

## Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import json
import time
import requests
import difflib
from collections import defaultdict
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# Google Colab setup (uncomment if needed)
# from google.colab import drive
# drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/DataDoseDepi'

## Part 1: Active Ingredient Cleaning

### Configuration & Constants

In [ ]:
# Encoded token decoder
ENCODED_TOKEN_MAP = {
    "__ING0024__": "vita",
    "__ING0035__": "",
    "__ING0055__": "iron",
}

# Garbage terms (exact match only)
GARBAGE_LIST = [
    "invalid", "test", "unknown", "no active ingredient", "pending", "deleted",
    "n/a", "not available", "natural source", "mixed",
    "coming soon", "special", "bitten", "mental",
    "herbal formula", "regurgitation milk formula", "gasmin odor",
    "ethyhexyl", "stearly alcohol", "silicones",
    "glycerol stearate", "octadeceny ammonium",
    "distearoylethyl hydroxyethylmonium methosulfate",
    "capramidopropylbetaine", "capryl", "cocoamidopropyl betaine",
    "water", "aqua",
    "dr ey t", "dr ey",
    "vitamins", "vita", "350m",
]
GARBAGE_EXACT = set(x.strip().lower() for x in GARBAGE_LIST)

GARBAGE_TOKENS_EXACT = {
    "na", "n/a", "amin", "amins", "type", "formula",
    "other", "high", "pre", "mixed", "special",
    "as", "ivay", "potat", "len",
    "2",
    "vitamin",
}

# Non-drug tokens (removed as individual tokens)
NON_DRUG_TOKENS = {
    "q10", "coq10", "q 10",
    "royal jelly", "propolis", "bee pollen", "bee wax",
    "antioxidants", "antioxidant",
    "honey", "beeswax", "aloe vera", "aloe",
    "turmeric", "curcumin", "ginger", "ginger extract",
    "garlic", "garlic extract", "garlic powder",
    "cinnamon", "cinnamon extract",
    "msm", "methylsulfonylmethane",
    "spirulina", "chlorella",
    "milk thistle",
}

# Spell-fix dictionary (typos only, no synonym merging)
SPELL_FIX = {
    "benzylpenicillin sodiium": "benzylpenicillin sodium",
    "cholorohexidine": "chlorhexidine",
    "chlorohexidine": "chlorhexidine",
    "chlorohexidin": "chlorhexidine",
    "nitrofurantion": "nitrofurantoin",
    "immunoglobulins": "immunoglobulin",
    "panthenoll": "panthenol",
    "pilocarpin": "pilocarpine",
    "macrophages": "macrophage",
    "macrofage": "macrophage",
    "olanzapin": "olanzapine",
    "diclofienac": "diclofenac",
    "sildeanfil": "sildenafil",
    "sindalfil": "sildenafil",
    "digoxine": "digoxin",
}

PLAIN_REPLACEMENTS = {
    "vit.": "vitamin ",
    "vitamin b complex": "thiamine + riboflavin + niacin + pantothenic acid + pyridoxine + biotin + folic acid + cobalamin",
    "b complex": "thiamine + riboflavin + niacin + pantothenic acid + pyridoxine + biotin + folic acid + cobalamin",
}

BVITAMIN_REGEX = [
    (re.compile(r'\bvit\b\.?'), "vitamin "),
    (re.compile(r'\bb12\b'), "cobalamin"),
    (re.compile(r'\bb9\b'), "folic acid"),
    (re.compile(r'\bb7\b'), "biotin"),
    (re.compile(r'\bb6\b'), "pyridoxine"),
    (re.compile(r'\bb5\b'), "pantothenic acid"),
    (re.compile(r'\bb3\b'), "niacin"),
    (re.compile(r'\bb2\b'), "riboflavin"),
    (re.compile(r'\bb1\b'), "thiamine"),
]

DOSE_UNIT_PATTERN = re.compile(
    r'\d+(\.\d+)?\s*(mg|g|gm|mcg|µg|ug|iu|i\s*u|miu|ml|%|units?|tabs?|caps?|amp|vial)\b',
    re.IGNORECASE
)

COSMETIC_TERMS = {
    "cream", "shampoo", "lotion", "hair", "gel",
    "serum", "moisturizer", "conditioner", "spray", "foam",
    "mask", "scrub", "toner", "cleanser", "balm",
    "fragrance", "deodorant", "sunscreen", "exfoliant",
    "skin", "whitening", "uva", "uvb",
}

VALID_VITAMIN_LETTERS = {"a", "c", "d", "e", "k", "d2", "d3", "k1", "k2", "k3",
    "b1", "b2", "b3", "b5", "b6", "b7", "b9", "b12"}

print("Configuration loaded.")

### Helper Functions: Text Normalization

In [ ]:
def decode_encoded_tokens(text):
    if not isinstance(text, str):
        return text
    for token, replacement in ENCODED_TOKEN_MAP.items():
        text = text.replace(token, replacement)
    text = re.sub(r'__[A-Z]+\d+__', ' ', text)
    return text

def apply_spell_fix(text):
    if not isinstance(text, str):
        return text
    for wrong, right in sorted(SPELL_FIX.items(), key=lambda x: -len(x[0])):
        if wrong in text:
            text = re.sub(r'\b' + re.escape(wrong) + r'\b', right, text)
    return text

def is_garbage_token(token):
    t = token.strip().lower()
    if not t or len(t) <= 2:
        return True
    if t in GARBAGE_EXACT or t in GARBAGE_TOKENS_EXACT or t in NON_DRUG_TOKENS:
        return True
    return False

def is_cosmetic_entry(text):
    if not text:
        return False
    words = set(text.lower().split())
    return len(words & COSMETIC_TERMS) >= 2

def normalize_omega(text):
    def expand_omega_multi(m):
        nums = m.group(1).split('-')
        return ' + '.join('omega ' + n for n in nums)
    text = re.sub(r'\bomega-(\d+(?:-\d+)+)\b', expand_omega_multi, text)
    text = re.sub(r'\bomega-(\d+)\b', r'omega \1', text)
    return text

def normalize_text(text):
    if pd.isna(text) or not isinstance(text, str):
        return None
    
    text = text.strip()
    if not text or text.lower() in GARBAGE_EXACT:
        return None
    
    text = text.lower()
    text = apply_spell_fix(text)
    
    for wrong, right in PLAIN_REPLACEMENTS.items():
        if wrong in text:
            text = text.replace(wrong, right)
    
    for pattern, right in BVITAMIN_REGEX:
        text = pattern.sub(right, text)
    
    text = normalize_omega(text)
    text = re.sub(r'\bvit\.?\s+', 'vitamin ', text)
    
    text = re.sub(r'(--|–|-|/|,|;|\|&| and | with |&|\+)', ' + ', text)
    text = DOSE_UNIT_PATTERN.sub(' ', text)
    text = re.sub(r'\b\d{2,}\b', ' ', text)
    text = re.sub(r'[^a-z0-9+\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'\s*\+\s*', ' + ', text).strip()
    text = text.strip('+ ')
    
    return text if text else None

### Helper Functions: Ingredient Validation

In [ ]:
def clean_ingredient_list(parts, original_text):
    cleaned = []
    
    for ingredient in parts:
        ingredient = re.sub(r'\s+', ' ', ingredient).strip()
        ingredient = re.sub(r'\b([a-z]{4,})\d+$', r'\1', ingredient)
        ingredient = re.sub(r'\s+', ' ', ingredient).strip()
        
        if is_garbage_token(ingredient) or len(ingredient) <= 2:
            continue
        
        cleaned.append(ingredient)
    
    return sorted(set(cleaned))

def clean_active_ingredient(text):
    text = decode_encoded_tokens(text)
    normalized = normalize_text(text)
    
    if normalized is None:
        return None
    
    if is_cosmetic_entry(normalized):
        return None
    
    parts = [p.strip() for p in normalized.split('+')]
    cleaned_parts = clean_ingredient_list(parts, normalized)
    
    if not cleaned_parts:
        return None
    
    return " + ".join(cleaned_parts)

### Stage 1: Clean Raw Ingredients

In [ ]:
INPUT_FILE = os.path.join(BASE_DIR, 'DataDoseDataset.csv')
CLEANED_CSV = os.path.join(BASE_DIR, 'DataDoseDataset_ActiveIngredient_Cleaned.csv')

df = pd.read_csv(INPUT_FILE)
print(f"Loaded: {len(df):,} rows")
print(f"Columns: {df.columns.tolist()}")

# Find ingredient column
col_name = None
possible_cols = ['activeingredient', 'ActiveIngredient', 'active_ingredient', 'ingredients', 'Ingredients']
for col in possible_cols:
    if col in df.columns:
        col_name = col
        break

if col_name:
    print(f"\nUsing ingredient column: '{col_name}'")
    
    df['activeingredient_clean'] = df[col_name].apply(clean_active_ingredient)
    
    df_valid = df[df['activeingredient_clean'].notna()].reset_index(drop=True)
    
    print(f"\nCleaning results:")
    print(f"  Valid rows: {len(df_valid):,} / {len(df):,}")
    print(f"  Removed: {len(df) - len(df_valid):,} rows")
    print(f"\nSample cleaned ingredients:")
    print(df_valid['activeingredient_clean'].head(10).to_string())
    
    df_valid.to_csv(CLEANED_CSV, index=False)
    print(f"\nSaved to: {CLEANED_CSV}")
else:
    print("ERROR: Ingredient column not found")

## Part 2: Ingredient Verification Against Reference

### LLM Configuration

In [ ]:
GROQ_API_KEYS = []  # Add your Groq API keys here
OPENROUTER_API_KEYS = []  # Add your OpenRouter API keys here
GROQ_MODEL = "llama-3.1-8b-instant"
OPENROUTER_MODEL = "meta-llama/llama-3.1-8b-instruct"

_SYSTEM_PROMPT = """You are a senior clinical pharmacist specializing in drug nomenclature.
Your task: Return ONLY the canonical INN name for the given ingredient.
- Correct misspellings (e.g., amoxcillin → amoxicillin)
- Canonicalize synonyms to INN (e.g., acetaminophen → paracetamol)
- Strip salts/esters (e.g., diclofenac sodium → diclofenac)
- For combinations, separate by ' + ' and sort alphabetically
- If not a drug ingredient, return UNKNOWN
- Return ONLY the INN, no explanation."""

print("LLM configuration ready.")

### Fuzzy Matching Against Reference

In [ ]:
def fuzzy_match_ingredient(ingredient, reference_list, threshold=0.85):
    best_match = None
    best_score = 0
    
    for ref_ing in reference_list:
        ratio = difflib.SequenceMatcher(None, ingredient.lower(), ref_ing.lower()).ratio()
        if ratio > best_score:
            best_score = ratio
            best_match = ref_ing if ratio >= threshold else None
    
    return best_match, best_score if best_match else 0

def verify_ingredients(cleaned_csv, reference_csv=None, output_csv=None):
    df = pd.read_csv(cleaned_csv)
    print(f"Loaded cleaned data: {len(df):,} rows")
    
    # Extract unique ingredients
    all_ingredients = []
    for val in df['activeingredient_clean'].dropna():
        parts = [p.strip() for p in str(val).split('+')]
        all_ingredients.extend(parts)
    
    unique_ingredients = sorted(set(all_ingredients))
    print(f"Unique ingredients: {len(unique_ingredients):,}")
    
    # Load reference if provided
    reference_list = []
    if reference_csv and os.path.exists(reference_csv):
        ref_df = pd.read_csv(reference_csv)
        if 'ingredient' in ref_df.columns:
            reference_list = ref_df['ingredient'].dropna().tolist()
            print(f"Reference ingredients: {len(reference_list):,}")
    
    # Verify each ingredient
    verification_results = []
    for ing in unique_ingredients[:100]:  # Process first 100 for demo
        match, score = fuzzy_match_ingredient(ing, reference_list)
        verification_results.append({
            'ingredient': ing,
            'reference_match': match,
            'match_score': score,
        })
    
    verify_df = pd.DataFrame(verification_results)
    if output_csv:
        verify_df.to_csv(output_csv, index=False)
        print(f"\nVerification results saved to: {output_csv}")
    
    print(f"\nSample verification results:")
    print(verify_df.head(10).to_string())
    
    return verify_df

# Example call (uncomment to run)
# verify_df = verify_ingredients(CLEANED_CSV)

## Part 3: Export Final Dataset

In [ ]:
# Final dataset is ready with cleaned active ingredient column
# Add verification columns as needed based on reference data

print(f"Active ingredient cleaning complete.")
print(f"Cleaned data columns: {df_valid.columns.tolist()}")
print(f"\nNext steps:")
print("  1. Load reference ingredient data if available")
print("  2. Run verify_ingredients() for fuzzy matching")
print("  3. Use LLM verification for unmatched ingredients")
print("  4. Export final verified dataset")